In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


**0. Leitura do Dataset para uma dataframe Pandas**

In [ ]:
# Leitura do dataset. O ficheiro 'ObesityDataset.csv' deve estar colocado na diretoria de trabalho

# Esta é uma versão corrigida do dataset original que está disponível aqui: https://www.kaggle.com/datasets/ezzaldeenesmail/obesitydataset-raw-and-data-sinthetic
# 1. Foram eliminadas entradas duplicadas
# 2. A parte decimal dos atriburos numéricos foi eliminada

# O dataset fica armazenado numa dataframe Pandas chamada df

df= pd.read_csv('ObesityDataset.csv')


**1. Visão Global do Dataset / Caracterização**

In [ ]:
# Dimensão do dataset:
# Número de Exemplos / Atributos

print('Dimensão: ', df.shape)

print('Número de exemplos: ', df.shape[0])
print('Número de atributos: ', df.shape[1])



In [ ]:
# Visualizar as primeiras 20 linhas

df.head(20)

In [ ]:
# Alterar o nome da coluna Target

df = df.rename(columns={'NObeyesdad': 'Obesity'})

In [ ]:
# Visualizar um resumo da informação armazenada
# 1. Número de exemplos
# 2. Número de atributos
# 3. Tipo de atributos
# 4. Valores em falta

# Existe inconsistência na forma de representar atributos categóricos ordinais
# Esta inconsistência será tratada mais tarde

df.info()

In [ ]:
# Distribuição dos exemplos pelas classes do target

# Verificar se estão equilibradas

ordem_obesity = ["Insufficient_Weight", "Normal_Weight",
         "Overweight_Level_I", "Overweight_Level_II",
         "Obesity_Type_I", "Obesity_Type_II", "Obesity_Type_III"]

df["Obesity"].value_counts(normalize=True).reindex(ordem_obesity).mul(100)

**2. Pré-processamento de Dados**

**2.1 Limpar Dados**

**2.1.1 Valores em Falta**

In [ ]:
# Pesquisar por dados em falta

print(df.isnull().sum())

print('Total de dados em falta: ', df.isnull().sum().sum())


**2.1.2. Deteção de Outliers**

In [ ]:

# Para atributos numéricos podemos utilizar o método describe()
# Não efetua a definição matemática habitual de outlier. É apenas uma inspeção visual

df.describe()

In [ ]:
# Verificação da gama de valores em atributos categóricos

cat = df.select_dtypes(include=["object", "string", "category"]).columns

for c in cat:
    print("\n", df[c].value_counts().to_string())

**2.2. Transformar Dados**

In [ ]:
# Existe inconsistência na forma de representar e identificar atributos categóricos
# Alguns estão identificados como numéricos e outros como objetos

# Aplica-se uma transformação e todos passam a ser considerados numéricos, o que vai simplificar a análise

# Mapas de codificação (valor original → código)
bin_map    = {'no': 0, 'yes': 1}
gender_map = {'Female': 0, 'Male': 1}
freq_map   = {'no': 0, 'Sometimes': 1, 'Frequently': 2, 'Always': 3}
obesity_map = {k: i for i, k in enumerate(ordem_obesity)}

# Mapas inversos (código → valor original), para gráficos e interpretação
bin_inv     = {v: k for k, v in bin_map.items()}
gender_inv  = {v: k for k, v in gender_map.items()}
freq_inv    = {v: k for k, v in freq_map.items()}
obesity_inv = {v: k for k, v in obesity_map.items()}

# Aplicar

# Mapear os atributos binários:
for c in ['family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC']:
    df[c] = df[c].map(bin_map)

df['Gender'] = df['Gender'].map(gender_map)

# Mapear os atributos com 4 possibilidades: No; Sometimes; Frequently; Always
for c in ['CAEC', 'CALC']:
    df[c] = df[c].map(freq_map)

# Mapear o target
df['Obesity'] = df['Obesity'].map(obesity_map)

# Mapear o atributo categórico não ordinal (codificação one-hot)
df = pd.get_dummies(df, columns=['MTRANS'], dtype=int)


In [ ]:
df.head()

In [ ]:
df.info()

**3. Visualização de Dados**

**3.1 - Visualização Univariada**

In [ ]:
# Separar atributos (variáveis dependentes) do target (classe dos exemplos)
# A partir desta célula só iremos trabalhar com os dataframes Att e Classe

Att = df.drop(['Obesity'], axis=1)
Classe = df['Obesity']

print('Target: ', Classe.shape)
print('Atributos: ', Att.shape)

In [ ]:
# Resumo estatístico dos dados armazenados
# Média, Desvio padrão, Mínimo, Máximo, Primeiro, Segundo (Mediana) e Terceiro Quartil

Att.describe()

In [ ]:
# Criar histogramas para os atributos que são contínuos
# O número de intervalos influencia a distribuição visualizada

bins = 5

Select_Hist = ["Age", "Height", "Weight"]

Att[Select_Hist].hist(bins=bins, figsize= (4*len(Select_Hist),4), layout=(1, len(Select_Hist)))
plt.show()

In [ ]:
# visualizar o tipo de transporte

mtrans_cols = [c for c in Att.columns if c.startswith('MTRANS_')]
contagens = Att[mtrans_cols].sum().sort_values()
contagens.index = contagens.index.str.removeprefix('MTRANS_')

ax = contagens.plot.bar(figsize=(7, 4), edgecolor='white')
ax.bar_label(ax.containers[0], padding=3)
ax.set_ylabel('Nº de indivíduos')
ax.set_ylim(0, contagens.max() * 1.1)   # espaço para o rótulo da maior barra
plt.tight_layout()
plt.show()




In [ ]:
# Criar Gráfico de barras com a distribuição do Target

contagens = Classe.value_counts().sort_index()          # códigos 0–6 por ordem
contagens.index = [ordem_obesity[i] for i in contagens.index]

contagens.plot.bar(figsize=(8, 4), edgecolor='white')
plt.ylabel('Total por classe')
plt.show()

In [ ]:
# Criar Boxplots para alguns atributos

Select_Box = ["Age", "Height", "Weight", "CALC"]

Att[Select_Box].plot.box(subplots=True,
                         layout=(1, len(Select_Box)),
                         figsize=(4 * len(Select_Box), 4))
plt.show()


In [ ]:
# Visualizar os boxplots dos atributos selecionados divididos por sexo

# É criada uma dataframe temporária para se visualizar o eixo xx com a identificação original

plot_df = Att.assign(Gender=Att['Gender'].replace(gender_inv))

axes = plot_df.boxplot(column=Select_Box, by='Gender', layout=(1, len(Select_Box)),
                       sharey=False, figsize=(4 * len(Select_Box), 4))
plt.tight_layout()
plt.show()

**3.2 - Visualização Multivariada**

In [ ]:
# Matriz de Correlação entre todos os atributos independentes
# Utiliza o método Spearman

corr = Att.corr(method="spearman")
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

plt.figure(figsize=(14, 12))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, square=True)
plt.title("Spearman - todos os atributos")
plt.show()




In [ ]:
# O que estamos a ver aqui?

mtrans_cols = [c for c in Att.columns if c.startswith('MTRANS_')]

# cada linha tem exatamente um 1; idxmax devolve o nome da coluna onde está
mtrans = Att[mtrans_cols].idxmax(axis=1).str.removeprefix('MTRANS_')

plot_df = Att.assign(MTRANS=mtrans)

ordem = plot_df.groupby('MTRANS')['Age'].median().sort_values().index
plot_df['MTRANS'] = pd.Categorical(plot_df['MTRANS'], categories=ordem, ordered=True)

plot_df.boxplot(column='Age', by='MTRANS', figsize=(9, 4), rot=30)
plt.suptitle('')
plt.title('Idade por meio de transporte')
plt.xlabel('')
plt.ylabel('anos')
plt.tight_layout()
plt.show()

In [ ]:
# Correlação individual de cada atributo independente com a variável dependente

rho = Att.corrwith(Classe, method='spearman').sort_values(key=abs)
rho.plot.barh(figsize=(6, 7))
plt.axvline(0, color='grey', lw=0.8)
plt.xlabel('Correlação Spearman com a variável dependente')
plt.tight_layout()
plt.show()

In [ ]:
# Criar matrizes de dispersão
# Matriz de dispersão: um gráfico para cada par de atributos selecionados
# Na diagonal surge a distribuição de cada atributo

Att_Scatter = ["Age", "Height", "Weight", "NCP"]

sns.pairplot(Att[Att_Scatter], plot_kws={"s": 10, "alpha": 0.5}, corner=True)
plt.show()

In [ ]:

# Scatter plots com informação sobre 2 atributos

# Escolher atributos

A1 = 'Height'
A2 = 'Weight'

# Criar o gráfico

sns.scatterplot(data=Att, x=A1, y=A2, s=10)
plt.tight_layout()

plt.show()


In [ ]:


# Scatter Plot com 3 atributos
# A terceira dimensão (cor) representa o género

sns.scatterplot(data=Att, x='Height', y='Weight', s=10,
                hue=Att['Gender'].map(gender_inv), palette='turbo')

plt.legend(title='Gender', bbox_to_anchor=(1, 1))
plt.show()

In [ ]:

# Scatter Plot com 3 atributos
# A terceira dimensão (cor) representa as várias classes do target


A1 = 'Height'
A2 = 'Weight'

sns.scatterplot(data=Att, x=A1, y=A2, s=10,
                hue=Classe.map(obesity_inv), hue_order=ordem_obesity, palette='turbo')

plt.legend(title='Obesity', bbox_to_anchor=(1, 1))
plt.show()


In [ ]:
# Scatter Plot com 4 atributos
# A terceira dimensão (cor) representa as várias classes
# A quarta dimensão representa o género


A1 = 'Height'
A2 = 'Weight'

sns.scatterplot(data=Att, x=A1, y=A2, s=15,
                hue=Classe.map(obesity_inv), hue_order=ordem_obesity, palette='turbo',
                style=Att['Gender'].map(gender_inv), markers={'Female': 'o', 'Male': '^'})
plt.legend(bbox_to_anchor=(1, 1))
plt.show()